
# Patch-only: preserve your original analysis/visualization cells

不要用这个 notebook 替换你的原 notebook。

用法：

1. 打开你原来的 notebook。
2. 保留你原来的所有分析、统计、可视化 cell。
3. 找到原来定义 `infer_case_metadata`、`make_args_from_case`、`build_trainer_for_case` 的 cell。
4. 在那个 cell 后面，新建一个 cell。
5. 把下面这个 patch cell 粘进去运行。
6. 再运行你原来的 build 循环和后续可视化。

这个 patch 只覆盖：
- `infer_case_metadata`
- `make_args_from_case`
- `build_trainer_for_case`

不会覆盖你的分析函数、绘图函数、结果表、可视化代码。


In [ ]:

# ============================================================
# Patch cell: only fix config/build; keep your analysis/visualization cells
# ============================================================

import ast
import sys
import inspect
import importlib
import pkgutil
from pathlib import Path
from types import SimpleNamespace

import torch
import pandas as pd
import numpy as np

# ------------------------------------------------------------
# 0. Ensure paths and project setup_cfg are available
# ------------------------------------------------------------

if "REPO_ROOT" not in globals():
    raise RuntimeError("请先运行你原 notebook 的初始化 cell，确保 REPO_ROOT 已定义")
if "DATA_ROOT" not in globals():
    DATA_ROOT = Path(REPO_ROOT) / "DATASETS"

REPO_ROOT = Path(REPO_ROOT).expanduser().resolve()
DATA_ROOT = Path(DATA_ROOT).expanduser().resolve()

# optional
if "DASSL_ROOT" in globals():
    DASSL_ROOT = Path(DASSL_ROOT).expanduser().resolve()
else:
    DASSL_ROOT = Path("/root/autodl-tmp/Dassl.pytorch").expanduser().resolve()

for p in [REPO_ROOT, DASSL_ROOT]:
    p = str(Path(p).resolve())
    if p not in sys.path:
        sys.path.insert(0, p)

try:
    import tools.train as train_mod
    print("Loaded tools.train from:", inspect.getfile(train_mod))
except Exception as e:
    raise RuntimeError(f"无法导入 tools.train；请检查 REPO_ROOT/sys.path: {repr(e)}")

from dassl.engine import build_trainer, TRAINER_REGISTRY

# Trigger registry import, but do not touch your analysis functions.
def _import_all_submodules(package_name, max_depth=6):
    try:
        package = importlib.import_module(package_name)
    except Exception as e:
        print(f"[WARN] Cannot import package {package_name}: {repr(e)}")
        return

    if not hasattr(package, "__path__"):
        return

    prefix = package.__name__ + "."
    for module_info in pkgutil.walk_packages(package.__path__, prefix=prefix):
        full_name = module_info.name
        depth = full_name.count(".") - package_name.count(".")
        if depth > max_depth:
            continue
        try:
            importlib.import_module(full_name)
        except Exception as e:
            print(f"[WARN] Cannot import {full_name}: {repr(e)}")

_import_all_submodules("datasets")
_import_all_submodules("trainers")

print("Registered trainers:", TRAINER_REGISTRY.registered_names())


# ------------------------------------------------------------
# 1. Conversion helpers
# ------------------------------------------------------------

def decode_backbone_name(x):
    mapping = {
        "ViT-B-16": "ViT-B/16",
        "ViT-B-32": "ViT-B/32",
        "ViT-L-14": "ViT-L/14",
        "ViT-L-14-336px": "ViT-L/14@336px",
    }
    return mapping.get(str(x), str(x))


def encode_backbone_folder(x):
    mapping = {
        "ViT-B/16": "ViT-B-16",
        "ViT-B/32": "ViT-B-32",
        "ViT-L/14": "ViT-L-14",
        "ViT-L/14@336px": "ViT-L-14-336px",
    }
    return mapping.get(str(x), str(x))


def canonical_dataset_name(x):
    mapping = {
        "caltech101": "Caltech101",
        "oxford_pets": "OxfordPets",
        "stanford_cars": "StanfordCars",
        "oxford_flowers": "OxfordFlowers",
        "food101": "Food101",
        "fgvc_aircraft": "FGVCAircraft",
        "sun397": "SUN397",
        "dtd": "DescribableTextures",
        "eurosat": "EuroSAT",
        "ucf101": "UCF101",
        "imagenet": "ImageNet",
    }
    return mapping.get(str(x), str(x))


def _set_opt(opts, key, value):
    opts = [] if opts is None else list(opts)
    key = str(key)
    value = str(value)

    if key in opts:
        idx = opts.index(key)
        if idx + 1 >= len(opts):
            raise ValueError(f"opts 中 {key} 后面没有 value")
        opts[idx + 1] = value
    else:
        opts.extend([key, value])
    return opts


# ------------------------------------------------------------
# 2. Parse training log to avoid guessing runner/method/opts
# ------------------------------------------------------------

def parse_log_txt(case_root):
    case_root = Path(case_root)
    log_path = case_root / "log.txt"

    out = {
        "log_path": str(log_path) if log_path.exists() else "",
        "log_trainer": None,
        "log_method": None,
        "log_opts": None,
    }

    if not log_path.exists():
        return out

    txt = log_path.read_text(errors="ignore")

    for raw in txt.splitlines():
        line = raw.strip()

        if line.startswith("method:"):
            out["log_method"] = line.split(":", 1)[1].strip()

        elif line.startswith("trainer:"):
            out["log_trainer"] = line.split(":", 1)[1].strip()

        elif line.startswith("opts:"):
            s = line.split(":", 1)[1].strip()
            try:
                val = ast.literal_eval(s)
                if isinstance(val, list):
                    out["log_opts"] = [str(x) for x in val]
            except Exception as e:
                print("[WARN] parse opts failed:", log_path, repr(e))

    return out


def find_dataset_config_file(dataset):
    dataset = str(dataset)
    canonical = canonical_dataset_name(dataset)
    candidates = [
        REPO_ROOT / "configs" / "datasets" / f"{dataset}.yaml",
        REPO_ROOT / "configs" / "datasets" / f"{dataset.lower()}.yaml",
        REPO_ROOT / "configs" / "datasets" / f"{canonical}.yaml",
        REPO_ROOT / "configs" / "datasets" / f"{canonical.lower()}.yaml",
    ]
    for p in candidates:
        if p.exists():
            return str(p)
    return ""


# ------------------------------------------------------------
# 3. Override metadata and args only
# ------------------------------------------------------------

def infer_case_metadata(case_root, case=None):
    case = case or {}
    case_root = Path(case_root).expanduser().resolve()
    parts = case_root.parts

    anchors = [
        i for i, p in enumerate(parts)
        if p == "FS" and i + 1 < len(parts) and parts[i + 1] == "fewshot_train"
    ]
    if not anchors:
        raise ValueError(f"无法从路径中找到 FS/fewshot_train: {case_root}")

    i = anchors[-1]

    method_from_path = parts[i - 1]
    dataset_from_path = parts[i + 2]
    shots_str = parts[i + 3]
    backbone_folder = parts[i + 4]
    config = parts[i + 5]
    seed_str = parts[i + 6]

    shots = int(shots_str.replace("shots_", ""))
    seed = int(seed_str.replace("seed", ""))

    output_family = "output_refactor" if "output_refactor" in parts else (
        "output_sweeps" if "output_sweeps" in parts else "unknown"
    )

    log_info = parse_log_txt(case_root)

    method = log_info.get("log_method") or case.get("method") or method_from_path

    # Most important distinction:
    # trainer is Dassl runner registry name, e.g. RefactorRunner.
    # method is the BayesMMRL method being analyzed.
    trainer = log_info.get("log_trainer") or case.get("trainer") or "RefactorRunner"

    backbone = decode_backbone_name(case.get("backbone", decode_backbone_name(backbone_folder)))
    dataset = case.get("dataset", dataset_from_path)

    meta = {
        "name": case.get("name", case_root.name),
        "case_root": case_root,
        "output_family": output_family,

        "method_from_path": method_from_path,
        "method": method,
        "trainer": trainer,

        "dataset": dataset,
        "dataset_name": canonical_dataset_name(dataset),
        "shots": int(case.get("shots", shots)),

        "backbone_folder": encode_backbone_folder(backbone),
        "backbone": backbone,

        "config": case.get("config", config),
        "seed": int(case.get("seed", seed)),

        "log_path": log_info.get("log_path"),
        "log_trainer": log_info.get("log_trainer"),
        "log_method": log_info.get("log_method"),
        "log_opts": log_info.get("log_opts"),

        "dataset_config_file": find_dataset_config_file(dataset),
    }

    return meta


def make_args_from_case(meta):
    case_root = Path(meta["case_root"]).resolve()
    args = SimpleNamespace()

    args.root = str(DATA_ROOT)
    args.output_dir = str(case_root)
    args.resume = ""
    args.seed = int(meta["seed"])

    args.source_domains = None
    args.target_domains = None
    args.transforms = None

    args.trainer = meta["trainer"]  # runner, e.g. RefactorRunner
    args.method = meta["method"]    # actual method, e.g. BayesMMRL
    args.backbone = meta["backbone"]
    args.head = ""

    args.dataset_config_file = meta.get("dataset_config_file", "")
    args.config_file = ""

    args.eval_only = True
    args.model_dir = str(case_root)
    args.load_epoch = None
    args.no_train = True

    # Preserve training-time BAYES_MMRL.* opts from log.txt.
    opts = list(meta.get("log_opts") or [])

    # Then correct only generic/base fields.
    opts = _set_opt(opts, "DATASET.ROOT", str(DATA_ROOT))
    opts = _set_opt(opts, "DATASET.NAME", meta["dataset_name"])
    opts = _set_opt(opts, "DATASET.NUM_SHOTS", int(meta["shots"]))
    opts = _set_opt(opts, "MODEL.BACKBONE.NAME", meta["backbone"])
    opts = _set_opt(opts, "SEED", int(meta["seed"]))
    opts = _set_opt(opts, "OUTPUT_DIR", str(case_root))

    args.opts = opts
    return args


def setup_cfg_for_case(args):
    # Prefer the project's own setup_cfg to keep implementation behavior unchanged.
    if hasattr(train_mod, "setup_cfg"):
        return train_mod.setup_cfg(args)

    # Fallback only if project has no setup_cfg.
    from dassl.config import get_cfg_default
    cfg = get_cfg_default()
    if hasattr(train_mod, "extend_cfg"):
        train_mod.extend_cfg(cfg)

    if getattr(args, "dataset_config_file", ""):
        cfg.merge_from_file(args.dataset_config_file)

    if getattr(args, "root", None):
        cfg.DATASET.ROOT = args.root
    if getattr(args, "output_dir", None):
        cfg.OUTPUT_DIR = args.output_dir
    if getattr(args, "seed", None) is not None:
        cfg.SEED = int(args.seed)
    if getattr(args, "trainer", None):
        cfg.TRAINER.NAME = args.trainer
    if getattr(args, "backbone", None):
        cfg.MODEL.BACKBONE.NAME = args.backbone

    if getattr(args, "opts", None):
        cfg.merge_from_list(args.opts)

    cfg.freeze()
    return cfg


def debug_print_build_inputs(meta, args, cfg):
    print("\n[PATCH DEBUG]")
    print("name                      =", meta["name"])
    print("case_root                 =", meta["case_root"])
    print("method                    =", meta["method"])
    print("trainer/runner            =", meta["trainer"])
    print("log_method                =", meta.get("log_method"))
    print("log_trainer               =", meta.get("log_trainer"))
    print("backbone_folder(path)     =", meta["backbone_folder"])
    print("backbone(cfg)             =", meta["backbone"])
    print("dataset(path)             =", meta["dataset"])
    print("dataset(cfg)              =", meta["dataset_name"])
    print("dataset_config_file       =", meta.get("dataset_config_file") or "<none>")
    print("cfg.TRAINER.NAME          =", cfg.TRAINER.NAME)
    print("cfg.DATASET.NAME          =", cfg.DATASET.NAME)
    print("cfg.MODEL.BACKBONE.NAME   =", cfg.MODEL.BACKBONE.NAME)

    if meta["log_opts"]:
        n_bayes = sum(1 for x in meta["log_opts"] if "BAYES_MMRL" in x)
        print("log opts count            =", len(meta["log_opts"]))
        print("BAYES_MMRL opts entries   =", n_bayes)

    assert cfg.TRAINER.NAME == meta["trainer"], (
        f"runner 不一致: cfg.TRAINER.NAME={cfg.TRAINER.NAME}, meta['trainer']={meta['trainer']}"
    )
    assert cfg.MODEL.BACKBONE.NAME == meta["backbone"], (
        f"backbone 不一致: cfg={cfg.MODEL.BACKBONE.NAME}, meta={meta['backbone']}"
    )


def build_trainer_for_case(case):
    case_root = Path(case["case_root"]).expanduser().resolve()
    meta = infer_case_metadata(case_root, case)
    args = make_args_from_case(meta)
    cfg = setup_cfg_for_case(args)

    debug_print_build_inputs(meta, args, cfg)

    print("\nBuilding runner:", cfg.TRAINER.NAME)
    print("Analyzing method:", meta["method"])

    trainer = build_trainer(cfg)

    print("\nLoading model from:", case_root)
    trainer.load_model(str(case_root))
    trainer.set_model_mode("eval")

    return trainer, meta


print("\nPatch applied.")
print("Now run your original build loop, then continue with your original analysis/visualization cells.")
